需要做的事情：
1、下载并准备好数据集，dexgraspNet，MultiDex，Unidexgrasp（已做），RealDex(已做)，DexGRAB
2、挨个训练上述还没做的数据集，保存模型
3、评估，保存实验数据
记录：
- A训练成功率、B评估成功率(总体的就行)，ckpt保存的路径
- C mlp的分类准确率 oracle，uniform，base000
- D base准确率，最后的提升后的成功率





cd /mnt/data1/zju/zhangziling/AAAI2027_Grasp/DemoGrasp_mine
source /mnt/data1/zju/wangyan/code/grasp/DemoGrasp_mine/.venv/bin/activate
# 抽取新的数据集物体 0
cd /mnt/data1/zju/zhangziling/AAAI2027_Grasp

# 处理realdex 数据集
python3 tools/datasets/prepare_dexgrasp_assets.py \
  --source-root /mnt/data1/zju/data/dexgrasp \
  --output-root DemoGrasp_mine/assets \
  --dataset all \
  --link-mode symlink \
  --overwrite


dexgrab/train.yaml
multidex/train.yaml
# 训练---A
CUDA_VISIBLE_DEVICES=1 python3 residual_se_grasp/train_se_reslearn.py \
  task=grasp train=PPOOneStep hand=new_sr_hand_simple \
  headless=true num_envs=512 \
  task.env.asset.multiObject=true \
  task.env.asset.multiObjectList="multidex/dexgrab.yaml" \
  +se_guide_mode=tilt_pitch \
  +se_tilt_angles=[0,15,30] \
  +se_tilt_axis=[1,0,0] \
  +se_pitch_angles=[0,15,30] \
  +se_sampling=random \
  checkpoint=/mnt/data1/zju/zhangziling/AAAI2027_Grasp/DemoGrasp_mine/runs_ppo/realdex_20260713_2225_multi_0.802/model_2700.pt \
  train.params.save_interval=100


# 评估--B
CUDA_VISIBLE_DEVICES=0 python3 pregrasp_eval/eval_yaw_SE_residual.py \
  task=grasp train=PPOOneStep hand=new_sr_hand_simple \
  test=True headless=False force_render=true \
  num_envs=32 \
  task.env.asset.multiObject=true \
  task.env.asset.multiObjectList="realdex/test.yaml" \
  checkpoint=runs_ppo/realdex_20260713_2344_multi_0.80/model_2400.pt \
  +pregrasp_yaw_angles=[0,30,60,90,120,150,180,210,240,270,300,330] \
  +pregrasp_yaw_angles=[0] \
  +se_tilt_angles=[0,15,30] \
  +se_pitch_angles=[0,15,30] \
  +pregrasp_yaw_sampling=cycle \
  +pregrasp_yaw_canonical_obs=True \
  +pregrasp_yaw_canonical_obs_scope=scene \
  +pregrasp_yaw_direct_base_qpos=True




## Realdex   1
### 采集rollout
CUDA_VISIBLE_DEVICES=1 python3 -m build_Pregrasp_data.yaw_pregrasp.collect.collect_yaw_pregrasp_rollouts \
  task=grasp train=PPOOneStep hand=new_sr_hand_simple \
  test=True headless=true force_render=false \
  num_envs=3200 \
  task.env.asset.multiObject=true \
  task.env.asset.multiObjectList="realdex/train.yaml" \
  checkpoint=runs_ppo/realdex_20260713_2225_multi_0.802/model_2700.pt \
  +pregrasp_yaw_angles=[0,30,60,90,120,150,180,210,240,270,300,330] \
  +se_tilt_angles=[0,15,30] \
  +se_pitch_angles=[0,15,30] \
  +pregrasp_yaw_frame=pca_short \
  +pregrasp_yaw_sampling=cycle \
  +pregrasp_yaw_canonical_obs=True \
  +pregrasp_yaw_canonical_obs_scope=scene \
  +pregrasp_yaw_direct_base_qpos=True \
  +yaw_pregrasp_collect_max_batches=100 \
+yaw_pregrasp_collect_rollout_root=/mnt/data1/zju/zhangziling/AAAI2027_Grasp/PregraspPrior/data/rollouts/realdex_m2700_yaw12_tilt3_pitch3_scene


### 聚合
python3 -m build_Pregrasp_data.yaw_pregrasp.aggregate.build_yaw_field_store \
  --rollout-root /mnt/data1/zju/zhangziling/AAAI2027_Grasp/PregraspPrior/data/rollouts/realdex_m2700_yaw12_tilt3_pitch3_scene \
  --field-root /mnt/data1/zju/zhangziling/AAAI2027_Grasp/PregraspPrior/data/fields/realdex_m2700_yaw12_tilt3_pitch3_scene


### 训练mlp
cd /mnt/data1/zju/zhangziling/AAAI2027_Grasp/PregraspPrior

PYTHONPATH=/mnt/data1/zju/zhangziling/AAAI2027_Grasp/PregraspPrior/src \
python3 -m oc_pregrasp.train.train_scorer \
  --rollout-root /mnt/data1/zju/zhangziling/AAAI2027_Grasp/PregraspPrior/data/rollouts/realdex_m2700_yaw12_tilt3_pitch3_scene \
  --output-dir /mnt/data1/zju/zhangziling/AAAI2027_Grasp/PregraspPrior/data/models/scorer_mlp_realdex_m2700_yaw12_tilt3_pitch3_scene \
  --feature-mode pca_short \
  --epochs 30 \
  --batch-size 2048 \
  --lr 0.001 \
  --device cuda

### 离线测试--------C
PYTHONPATH=/mnt/data1/zju/zhangziling/AAAI2027_Grasp/PregraspPrior/src \
python3 -m oc_pregrasp.eval.eval_scorer_offline \
  --checkpoint /mnt/data1/zju/zhangziling/AAAI2027_Grasp/PregraspPrior/data/models/scorer_mlp_realdex_m2700_yaw12_tilt3_pitch3_scene/best.pt \
  --rollout-root /mnt/data1/zju/zhangziling/AAAI2027_Grasp/PregraspPrior/data/rollouts/realdex_m2700_yaw12_tilt3_pitch3_scene \
  --output-json /mnt/data1/zju/zhangziling/AAAI2027_Grasp/PregraspPrior/data/evals/scorer_mlp_realdex_m2700_yaw12_tilt3_pitch3_scene_offline.json \
  --batch-size 1024 \
  --device cuda



### 测试mlp的预抓取操作（记录需要）----------D
cd /mnt/data1/zju/zhangziling/AAAI2027_Grasp/DemoGrasp_mine

CUDA_VISIBLE_DEVICES=0 python3 pregrasp_eval/eval_yaw_scorer_SE_residual.py \
  task=grasp train=PPOOneStep hand=new_sr_hand_simple \
  test=True headless=true force_render=false \
  num_envs=3200 \
  task.env.asset.multiObject=true \
  task.env.asset.multiObjectList="realdex/test.yaml" \
  checkpoint=runs_ppo/realdex_20260713_2225_multi_0.802/model_2700.pt \
  +pregrasp_yaw_scorer_checkpoint=/mnt/data1/zju/zhangziling/AAAI2027_Grasp/PregraspPrior/data/models/scorer_mlp_realdex_m2700_yaw12_tilt3_pitch3_scene/best.pt \
  +pregrasp_yaw_sampling=scorer_top1 \
  +pregrasp_yaw_frame=pca_short \
  +pregrasp_yaw_angles=[0,30,60,90,120,150,180,210,240,270,300,330] \
  +se_tilt_angles=[0,15,30] \
  +se_pitch_angles=[0,15,30] \
  +pregrasp_yaw_canonical_obs=True \
  +pregrasp_yaw_canonical_obs_scope=scene \
  +pregrasp_yaw_direct_base_qpos=True \
  +pregrasp_yaw_compare_base_rounds=2 \
  +pregrasp_yaw_compare_scorer_rounds=20 \
  +pregrasp_yaw_compare_base_mode=all \
  +pregrasp_yaw_compare_base_sampling=random \
  +pregrasp_eval_log_interval=0


## MultiDex  2
### 采集rollout
cd /mnt/data1/zju/zhangziling/AAAI2027_Grasp/DemoGrasp_mine

CUDA_VISIBLE_DEVICES=4 python3 -m build_Pregrasp_data.yaw_pregrasp.collect.collect_yaw_pregrasp_rollouts \
  task=grasp train=PPOOneStep hand=new_sr_hand_simple \
  test=True headless=true force_render=false \
  num_envs=3200 \
  task.env.asset.multiObject=true \
  task.env.asset.multiObjectList="multidex/train.yaml" \
  checkpoint=runs_ppo/multidex_20260715_0021/model_5300.pt \
  +pregrasp_yaw_angles=[0,30,60,90,120,150,180,210,240,270,300,330] \
  +se_tilt_angles=[0,15,30] \
  +se_pitch_angles=[0,15,30] \
  +pregrasp_yaw_frame=pca_short \
  +pregrasp_yaw_sampling=cycle \
  +pregrasp_yaw_canonical_obs=True \
  +pregrasp_yaw_canonical_obs_scope=scene \
  +pregrasp_yaw_direct_base_qpos=True \
  +yaw_pregrasp_collect_max_batches=100 \
  +yaw_pregrasp_collect_rollout_root=/mnt/data1/zju/zhangziling/AAAI2027_Grasp/PregraspPrior/data/rollouts/multidex_m5300_yaw12_tilt3_pitch3_scene


### 聚合
python3 -m build_Pregrasp_data.yaw_pregrasp.aggregate.build_yaw_field_store \
  --rollout-root /mnt/data1/zju/zhangziling/AAAI2027_Grasp/PregraspPrior/data/rollouts/multidex_m5300_yaw12_tilt3_pitch3_scene \
  --field-root /mnt/data1/zju/zhangziling/AAAI2027_Grasp/PregraspPrior/data/fields/multidex_m5300_yaw12_tilt3_pitch3_scene


### 训练mlp
cd /mnt/data1/zju/zhangziling/AAAI2027_Grasp/PregraspPrior

PYTHONPATH=/mnt/data1/zju/zhangziling/AAAI2027_Grasp/PregraspPrior/src \
python3 -m oc_pregrasp.train.train_scorer \
  --rollout-root /mnt/data1/zju/zhangziling/AAAI2027_Grasp/PregraspPrior/data/rollouts/multidex_m5300_yaw12_tilt3_pitch3_scene \
  --output-dir /mnt/data1/zju/zhangziling/AAAI2027_Grasp/PregraspPrior/data/models/scorer_mlp_multidex_m5300_yaw12_tilt3_pitch3_scene \
  --feature-mode pca_short \
  --epochs 30 \
  --batch-size 2048 \
  --lr 0.001 \
  --device cuda

### 离线测试--------C   
PYTHONPATH=/mnt/data1/zju/zhangziling/AAAI2027_Grasp/PregraspPrior/src \
python3 -m oc_pregrasp.eval.eval_scorer_offline \
  --checkpoint /mnt/data1/zju/zhangziling/AAAI2027_Grasp/PregraspPrior/data/models/scorer_mlp_multidex_m5300_yaw12_tilt3_pitch3_scene/best.pt \
  --rollout-root /mnt/data1/zju/zhangziling/AAAI2027_Grasp/PregraspPrior/data/rollouts/multidex_m5300_yaw12_tilt3_pitch3_scene \
  --output-json /mnt/data1/zju/zhangziling/AAAI2027_Grasp/PregraspPrior/data/evals/scorer_mlp_multidex_m5300_yaw12_tilt3_pitch3_scene_offline.json \
  --batch-size 1024 \
  --device cuda


### 测试mlp的预抓取操作（记录需要）----------D   0.84
cd /mnt/data1/zju/zhangziling/AAAI2027_Grasp/DemoGrasp_mine

CUDA_VISIBLE_DEVICES=6 python3 pregrasp_eval/eval_yaw_scorer_SE_residual.py \
  task=grasp train=PPOOneStep hand=new_sr_hand_simple \
  test=True headless=true force_render=false \
  num_envs=3200 \
  task.env.asset.multiObject=true \
  task.env.asset.multiObjectList="multidex/test.yaml" \
  checkpoint=runs_ppo/multidex_20260715_0021/model_5300.pt \
  +pregrasp_yaw_scorer_checkpoint=/mnt/data1/zju/zhangziling/AAAI2027_Grasp/PregraspPrior/data/models/scorer_mlp_multidex_m5300_yaw12_tilt3_pitch3_scene/best.pt \
  +pregrasp_yaw_sampling=scorer_top1 \
  +pregrasp_yaw_frame=pca_short \
  +pregrasp_yaw_angles=[0,30,60,90,120,150,180,210,240,270,300,330] \
  +se_tilt_angles=[0,15,30] \
  +se_pitch_angles=[0,15,30] \
  +pregrasp_yaw_canonical_obs=True \
  +pregrasp_yaw_canonical_obs_scope=scene \
  +pregrasp_yaw_direct_base_qpos=True \
  +pregrasp_yaw_compare_base_rounds=2 \
  +pregrasp_yaw_compare_scorer_rounds=20 \
  +pregrasp_yaw_compare_base_mode=all \
  +pregrasp_yaw_compare_base_sampling=random \
  +pregrasp_eval_log_interval=0



  base：0.93 
  mlp：0.97

## DexGRAB  3
### 采集rollout
cd /mnt/data1/zju/zhangziling/AAAI2027_Grasp/DemoGrasp_mine

CUDA_VISIBLE_DEVICES=5 python3 -m build_Pregrasp_data.yaw_pregrasp.collect.collect_yaw_pregrasp_rollouts \
  task=grasp train=PPOOneStep hand=new_sr_hand_simple \
  test=True headless=true force_render=false \
  num_envs=3200 \
  task.env.asset.multiObject=true \
  task.env.asset.multiObjectList="dexgrab/train.yaml" \
  checkpoint=runs_ppo/dexgrab_20260715_0021/model_5300.pt \
  +pregrasp_yaw_angles=[0,30,60,90,120,150,180,210,240,270,300,330] \
  +se_tilt_angles=[0,15,30] \
  +se_pitch_angles=[0,15,30] \
  +pregrasp_yaw_frame=pca_short \
  +pregrasp_yaw_sampling=cycle \
  +pregrasp_yaw_canonical_obs=True \
  +pregrasp_yaw_canonical_obs_scope=scene \
  +pregrasp_yaw_direct_base_qpos=True \
  +yaw_pregrasp_collect_max_batches=100 \
  +yaw_pregrasp_collect_rollout_root=/mnt/data1/zju/zhangziling/AAAI2027_Grasp/PregraspPrior/data/rollouts/dexgrab_m5300_yaw12_tilt3_pitch3_scene


### 聚合
python3 -m build_Pregrasp_data.yaw_pregrasp.aggregate.build_yaw_field_store \
  --rollout-root /mnt/data1/zju/zhangziling/AAAI2027_Grasp/PregraspPrior/data/rollouts/dexgrab_m5300_yaw12_tilt3_pitch3_scene \
  --field-root /mnt/data1/zju/zhangziling/AAAI2027_Grasp/PregraspPrior/data/fields/dexgrab_m5300_yaw12_tilt3_pitch3_scene


### 训练mlp
cd /mnt/data1/zju/zhangziling/AAAI2027_Grasp/PregraspPrior

PYTHONPATH=/mnt/data1/zju/zhangziling/AAAI2027_Grasp/PregraspPrior/src \
python3 -m oc_pregrasp.train.train_scorer \
  --rollout-root /mnt/data1/zju/zhangziling/AAAI2027_Grasp/PregraspPrior/data/rollouts/dexgrab_m5300_yaw12_tilt3_pitch3_scene \
  --output-dir /mnt/data1/zju/zhangziling/AAAI2027_Grasp/PregraspPrior/data/models/scorer_mlp_dexgrab_m5300_yaw12_tilt3_pitch3_scene \
  --feature-mode pca_short \
  --epochs 30 \
  --batch-size 2048 \
  --lr 0.001 \
  --device cuda

### 离线测试--------C  0.84
PYTHONPATH=/mnt/data1/zju/zhangziling/AAAI2027_Grasp/PregraspPrior/src \
python3 -m oc_pregrasp.eval.eval_scorer_offline \
  --checkpoint /mnt/data1/zju/zhangziling/AAAI2027_Grasp/PregraspPrior/data/models/scorer_mlp_dexgrab_m5300_yaw12_tilt3_pitch3_scene/best.pt \
  --rollout-root /mnt/data1/zju/zhangziling/AAAI2027_Grasp/PregraspPrior/data/rollouts/dexgrab_m5300_yaw12_tilt3_pitch3_scene \
  --output-json /mnt/data1/zju/zhangziling/AAAI2027_Grasp/PregraspPrior/data/evals/scorer_mlp_dexgrab_m5300_yaw12_tilt3_pitch3_scene_offline.json \
  --batch-size 1024 \
  --device cuda


### 测试mlp的预抓取操作（记录需要）----------D
cd /mnt/data1/zju/zhangziling/AAAI2027_Grasp/DemoGrasp_mine

CUDA_VISIBLE_DEVICES=5 python3 pregrasp_eval/eval_yaw_scorer_SE_residual.py \
  task=grasp train=PPOOneStep hand=new_sr_hand_simple \
  test=True headless=true force_render=false \
  num_envs=3200 \
  task.env.asset.multiObject=true \
  task.env.asset.multiObjectList="dexgrab/test.yaml" \
  checkpoint=runs_ppo/dexgrab_20260715_0021/model_5300.pt \
  +pregrasp_yaw_scorer_checkpoint=/mnt/data1/zju/zhangziling/AAAI2027_Grasp/PregraspPrior/data/models/scorer_mlp_dexgrab_m5300_yaw12_tilt3_pitch3_scene/best.pt \
  +pregrasp_yaw_sampling=scorer_top1 \
  +pregrasp_yaw_frame=pca_short \
  +pregrasp_yaw_angles=[0,30,60,90,120,150,180,210,240,270,300,330] \
  +se_tilt_angles=[0,15,30] \
  +se_pitch_angles=[0,15,30] \
  +pregrasp_yaw_canonical_obs=True \
  +pregrasp_yaw_canonical_obs_scope=scene \
  +pregrasp_yaw_direct_base_qpos=True \
  +pregrasp_yaw_compare_base_rounds=2 \
  +pregrasp_yaw_compare_scorer_rounds=20 \
  +pregrasp_yaw_compare_base_mode=all \
  +pregrasp_yaw_compare_base_sampling=random \
  +pregrasp_eval_log_interval=0


base:0.7
mlp:0.80

